# Visualizing ghost molecules

<video controls src="./assets/path_seeking_recording.webm">


Load the nanotube + methane example recording as an MDAnalsis universe:

In [1]:
from nanover.mdanalysis import universe_from_recording

universe = universe_from_recording("../recordings/nanotube-example-recording.nanover.zip")

In [2]:
from nanover.utilities.transforms import find_transformation_between_point_patterns, Transform

NANOTUBE_ATOMS = list(range(0, 60))
METHANE_ATOMS = list(range(60, 65))

_ = universe.trajectory[0]
NANOTUBE_INITIAL_POSITIONS = universe.atoms.positions[NANOTUBE_ATOMS] / 10  # angstrom -> nm

NANOTUBE_TRANSFORMS: list[Transform] = []
for _ in universe.trajectory:
    nanotube_positions = universe.atoms.positions[NANOTUBE_ATOMS] / 10  # angstrom -> nm
    matrix = find_transformation_between_point_patterns(NANOTUBE_INITIAL_POSITIONS, nanotube_positions)
    transform = Transform.from_parent_to_local_matrix(matrix)
    NANOTUBE_TRANSFORMS.append(transform)


GHOST_POSITION_SETS = []
def set_ghost_frames(*indexes):
    GHOST_POSITION_SETS.clear()
    for index in indexes:
        transform_i = NANOTUBE_TRANSFORMS[index]
        _ = universe.trajectory[index]
        methane_i_positions = universe.atoms.positions[METHANE_ATOMS] / 10  # angstrom -> nm
        methane_0_positions = [transform_i.point_local_to_parent(position) for position in methane_i_positions]
        GHOST_POSITION_SETS.append(methane_0_positions)

In [3]:
set_ghost_frames(0, 273, 610, 775, len(universe.trajectory)-1)

Set up the server with playback of the universe:

In [4]:
from nanover.app import OmniRunner
from nanover.mdanalysis import UniverseSimulation

simulation = UniverseSimulation.from_universe(universe, name="nanotube + methane")
simulation.playback_factor = 30
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: trajectory seeker")
imd_runner.load(0)

C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\coordinates\base.py:728: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt
C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\coordinates\base.py:728: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt


Import the jupyter utilities for drawing + interaction:

In [5]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

In [6]:
from nanover.trajectory import FrameData
from nanover.jupyter import FrameListener

class RelativeGhostVisuals(FrameListener):
    points = []

    def on_frame_update(self, full_frame: FrameData, frame_update: FrameData):
        frame_i_transform = NANOTUBE_TRANSFORMS[simulation.prev_frame]

        for s, positions in enumerate(GHOST_POSITION_SETS):
            for i, position in enumerate(positions):
                utilities.objects.update_shape(f"ghost.{s}.{i}", position=frame_i_transform.point_parent_to_local(position), size=0.1, color=[1.0, 1.0, 1.0, 0.25])

visuals = RelativeGhostVisuals.from_runner(imd_runner)
visuals.start()